In [1]:
%pip install faker
#cell 1

StatementMeta(, 9c4ddd1c-8aa3-4216-b3b3-9016ba76f1e4, 8, Finished, Available, Finished, True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 21.1 MB/s  0:00:00

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /nfs4/pyenv-d1f0e98d-86d1-483b-be23-3a3bad8e2f9f/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [ ]:
#CELL 2
# List of 6 warehouses: (id, name, city, state, postal code prefix, latitude, longitude, capacity)
warehouses = [
    ("WH01", "Sao Paulo Hub", "Sao Paulo", "SP", "01310", -23.5505, -46.6333, 50000),        # biggest hub
    ("WH02", "Rio Hub", "Rio de Janeiro", "RJ", "20040", -22.9068, -43.1729, 40000),         # Rio warehouse
    ("WH03", "Minas Hub", "Belo Horizonte", "MG", "30130", -19.9167, -43.9345, 30000),       # Minas Gerais warehouse
    ("WH04", "Parana Hub", "Curitiba", "PR", "80010", -25.4284, -49.2733, 25000),            # southern warehouse
    ("WH05", "Bahia Hub", "Salvador", "BA", "40020", -12.9777, -38.5016, 20000),             # northeast warehouse
    ("WH06", "Capital Hub", "Brasilia", "DF", "70040", -15.7939, -47.8828, 20000),           # central warehouse
]

# Column names for the table
cols = ["warehouse_id", "warehouse_name", "city", "state", "postal_code", "latitude", "longitude", "capacity_units"]

# Turn the Python list into a Spark DataFrame
wh_df = spark.createDataFrame(warehouses, cols)

# Save it as a Bronze Delta table (replace if it exists)
wh_df.write.mode("overwrite").format("delta").saveAsTable("bronze_warehouses")

# Show the result
display(wh_df)

In [4]:
import random        #CELL 3                           # Python's random number tools
from datetime import datetime, timedelta        # for working with dates and time gaps
from faker import Faker                         # the fake data library
from pyspark.sql import functions as F          # Spark helper functions

fake = Faker("pt_BR")                           # create a Faker generator (Brazilian style)
N_ORDERS = 200                                  # how many orders to create in this batch

# Today from 00:00 to 23:59: all fake orders get a purchase time in this window
batch_start = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
batch_end = batch_start + timedelta(hours=23, minutes=59)

# Helper: turn a datetime into text like "2026-10-17 12:30:00"
fmt = lambda d: d.strftime("%Y-%m-%d %H:%M:%S")

# Read real IDs from Bronze so fake data links to Olist data
warehouse_ids = [r.warehouse_id for r in spark.table("bronze_warehouses").collect()]                          # 6 warehouse IDs
customer_ids = [r.customer_id for r in spark.table("bronze_customers").orderBy(F.rand()).limit(N_ORDERS).collect()]  # 200 random customers
product_ids = [r.product_id for r in spark.table("bronze_products").select("product_id").collect()]          # all product IDs

carriers = ["Correios", "Jadlog", "Loggi", "Total Express"]   # fake carrier names
orders, shipments, depletions = [], [], []                    # three empty lists to fill

for i in range(N_ORDERS):                                                       # repeat once per order
    order_id = fake.uuid4().replace("-", "")                                    # unique order ID like Olist
    wh = random.choice(warehouse_ids)                                           # pick a random warehouse
    purchase = fake.date_time_between(start_date=batch_start, end_date=batch_end)  # random purchase time today
    approved = purchase + timedelta(minutes=random.randint(5, 120))             # approved 5-120 min later
    shipped = approved + timedelta(days=random.randint(1, 3))                   # handed to carrier 1-3 days later
    delivered = shipped + timedelta(days=random.randint(2, 12))                 # delivered 2-12 days after shipping
    estimated = shipped + timedelta(days=random.randint(5, 10))                 # promised delivery date
    orders.append((order_id, customer_ids[i], "delivered", fmt(purchase), fmt(approved), fmt(shipped), fmt(delivered), fmt(estimated)))  # one order row
    shipments.append((fake.uuid4(), order_id, wh, fmt(shipped), fmt(delivered), random.choice(carriers)))  # one shipment row
    depletions.append((fake.uuid4(), wh, random.choice(product_ids), random.randint(1, 5), random.randint(0, 200), fmt(purchase)))  # one stock drop row

# Orders: same columns as bronze_orders
order_cols = ["order_id", "customer_id", "order_status", "order_purchase_timestamp", "order_approved_at",
              "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"]
orders_df = spark.createDataFrame(orders, order_cols)                                          # list -> DataFrame
target_schema = spark.table("bronze_orders").schema                                            # copy column types from the Olist table
orders_df = orders_df.select([F.col(f.name).cast(f.dataType) for f in target_schema])          # match types and order exactly
orders_df.write.mode("append").format("delta").saveAsTable("bronze_orders")                    # APPEND under the Olist rows

# Shipments
ship_df = (spark.createDataFrame(shipments, ["shipment_id", "order_id", "warehouse_id", "shipped_at", "delivered_at", "carrier"])
           .withColumn("shipped_at", F.col("shipped_at").cast("timestamp"))                    # text -> timestamp
           .withColumn("delivered_at", F.col("delivered_at").cast("timestamp")))               # text -> timestamp
ship_df.write.mode("append").format("delta").saveAsTable("bronze_shipments")                   # append (creates table the first time)

# Stock depletions
dep_df = (spark.createDataFrame(depletions, ["depletion_id", "warehouse_id", "product_id", "quantity_depleted", "stock_remaining", "depletion_date"])
          .withColumn("depletion_date", F.col("depletion_date").cast("timestamp")))            # text -> timestamp
dep_df.write.mode("append").format("delta").saveAsTable("bronze_stock_depletions")             # append (creates table the first time)

print("Batch done:", len(orders), "orders,", len(shipments), "shipments,", len(depletions), "stock records")  # confirm

StatementMeta(, 7e53ec13-f4c1-410c-8072-fffbb0251977, 23, Finished, Available, Finished, False)

Batch done: 200 orders, 200 shipments, 200 stock records


In [5]:
print("orders:", spark.table("bronze_orders").count())    #CELL 4             # should be 99,441 + 200 = 99,641
print("shipments:", spark.table("bronze_shipments").count())           # should be 200
print("depletions:", spark.table("bronze_stock_depletions").count())   # should be 200

StatementMeta(, 7e53ec13-f4c1-410c-8072-fffbb0251977, 24, Finished, Available, Finished, False)

orders: 99641
shipments: 200
depletions: 200
